# module-modules-iter-isinstance-dispatch — faded example 3: Replace every ReLU with LeakyReLU in-place using named_modules dispatch

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `module-modules-iter-isinstance-dispatch`. The last cell reports your progress on the `GAN: model.modules() isinstance dispatch` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: model.modules() isinstance dispatch` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`module-modules-iter-isinstance-dispatch`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "module-modules-iter-isinstance-dispatch"
DD_SUBTOPIC = "GAN: model.modules() isinstance dispatch"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A common transfer-learning or architecture-search task is swapping one activation type for another across an entire network. `model.named_modules()` gives you both the qualified name and the module object, and `setattr` on the parent lets you replace a child by name. Because the qualified name uses dots as separators, you need to walk to the parent module before calling `setattr`.

## Faded exercise 3

Implement `replace_relu_with_leaky(model, negative_slope)` that replaces every `nn.ReLU` in `model` with `nn.LeakyReLU(negative_slope)` in-place, modifying the actual module tree.

Pattern:
```python
for name, m in model.named_modules():
    if isinstance(m, nn.ReLU):
        # split name to find parent + attr
        ...
        setattr(parent, attr_name, nn.LeakyReLU(negative_slope))
```

Helper already in scope: `def _get_module(model, dotted_name)` — walks the dot-separated path and returns the submodule at that path.

Your task: **fill in the replacement logic for when `isinstance(m, nn.ReLU)` is True**.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch
import torch.nn as nn

def _get_module(model, dotted_name):
    """Walk a dot-separated module path and return the module at that path."""
    parts = dotted_name.split('.')
    m = model
    for p in parts:
        m = getattr(m, p)
    return m

def replace_relu_with_leaky(model: nn.Module, negative_slope: float = 0.01) -> None:
    replacements = []
    for name, m in model.named_modules():
        if isinstance(m, nn.ReLU):
            replacements.append(name)
    for name in replacements:
        raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above

def _test():
    net = nn.Sequential(
        nn.Linear(8, 16),
        nn.ReLU(),
        nn.Linear(16, 8),
        nn.ReLU(),
        nn.Linear(8, 4),
    )
    replace_relu_with_leaky(net, negative_slope=0.1)
    for m in net.modules():
        assert not isinstance(m, nn.ReLU), "ReLU should have been replaced"
    leaky_mods = [m for m in net.modules() if isinstance(m, nn.LeakyReLU)]
    assert len(leaky_mods) == 2
    assert abs(leaky_mods[0].negative_slope - 0.1) < 1e-6


def _test():
    import torch
    import torch.nn as nn
    net = nn.Sequential(
        nn.Linear(8, 16),
        nn.ReLU(),
        nn.Linear(16, 8),
        nn.ReLU(),
        nn.Linear(8, 4),
    )
    replace_relu_with_leaky(net, negative_slope=0.1)
    for m in net.modules():
        assert not isinstance(m, nn.ReLU), "ReLU should have been replaced"
    leaky_mods = [m for m in net.modules() if isinstance(m, nn.LeakyReLU)]
    assert len(leaky_mods) == 2
    assert abs(leaky_mods[0].negative_slope - 0.1) < 1e-6


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch
import torch.nn as nn

def _get_module(model, dotted_name):
    parts = dotted_name.split('.')
    m = model
    for p in parts:
        m = getattr(m, p)
    return m

def replace_relu_with_leaky(model: nn.Module, negative_slope: float = 0.01) -> None:
    replacements = []
    for name, m in model.named_modules():
        if isinstance(m, nn.ReLU):
            replacements.append(name)
    for name in replacements:
        parts = name.rsplit('.', 1)
        if len(parts) == 1:
            # direct attribute of model
            setattr(model, name, nn.LeakyReLU(negative_slope))
        else:
            parent_name, attr_name = parts
            parent = _get_module(model, parent_name)
            setattr(parent, attr_name, nn.LeakyReLU(negative_slope))

def _test():
    net = nn.Sequential(
        nn.Linear(8, 16),
        nn.ReLU(),
        nn.Linear(16, 8),
        nn.ReLU(),
        nn.Linear(8, 4),
    )
    replace_relu_with_leaky(net, negative_slope=0.1)
    for m in net.modules():
        assert not isinstance(m, nn.ReLU), "ReLU should have been replaced"
    leaky_mods = [m for m in net.modules() if isinstance(m, nn.LeakyReLU)]
    assert len(leaky_mods) == 2
    assert abs(leaky_mods[0].negative_slope - 0.1) < 1e-6
```
</details>